# 🧠 Runtime Context & Tool-Mediated Access

---

## 🤔 `context_schema` vs `context` — what's the difference?

`context_schema=ColourContext` and `context=ColourContext()` look similar but serve **fundamentally different purposes**:

| Parameter | Role | Contains |
|---|---|---|
| `context_schema=ColourContext` | 📐 Type declaration | Just the class — no data |
| `context=ColourContext()` | 📦 Actual data | A live instance with real values |

> 💡 Think of it like a **function signature vs a function call**:

```python
# context_schema is like declaring the type
def agent(context: ColourContext): ...

# context= is like passing the actual argument
agent(context=ColourContext(favourite_colour="green"))
```

Even though `ColourContext` has default values (`favourite_colour="blue"`), those are just fallbacks on the class definition. The proof is in the last cell — passing `context=ColourContext(favourite_colour="green")` correctly overrides the default, which wouldn't matter if `context_schema` was already supplying the data.

> ✅ In a real app, `context_schema` would be a bare class with no defaults, and you'd always supply real per-user data via `context=` at invocation time — **same agent, different users, different context values each call**.

---

## 🔧 How context reaches the LLM

![Context via Tool Calls](resources/colour_context_tool_runtime.svg)

> ⚠️ The `context_schema` is **NOT** passed directly to the LLM. The LLM has **zero visibility** into the context object itself.

Instead, context is exposed **indirectly through tool calls**:
1. 🤖 LLM wants to know the user's favourite colour
2. 🔨 It calls the tool `get_favourite_colour`
3. ⚙️ The tool reads from `ToolRuntime` and returns the value
4. 💬 The LLM receives the result as a tool message

---

## ❓ Why not just dump the context into the system prompt?

The short answer: **context can get very large, very fast** 📈

In real applications, a user's context object might contain their full profile, preferences, history, permissions, account state, and more. Injecting all of that into every prompt has real costs:

| Problem | Impact |
|---|---|
| 🪙 **Token overhead** | Eats into the context window, less room for actual reasoning |
| 🌀 **Distraction** | "Lost in the middle" — models perform worse with bloated prompts |
| 🐢 **Latency & cost** | More tokens in = slower responses and higher API bills |
| 🔒 **Privacy** | LLM sees everything, even fields it has no business knowing |

> 🎯 The tool-call pattern solves this with **lazy, on-demand access** — the LLM only fetches what it actually needs, when it needs it.

This is the same principle behind **RAG** (Retrieval-Augmented Generation) — instead of stuffing a knowledge base into the prompt upfront, you retrieve only the relevant chunks at query time. 📚

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext  
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='0a8fb4d8-6147-4e75-9e97-c7702363cbb3'),
              AIMessage(content='I don’t know your favorite color yet. Want me to guess, or help you figure it out?\n\nIf you’d like a quick guess, blue is a common favorite—is that right? If not, tell me a bit about your preferences (for example, warm or cool colors, bright or muted tones, and what you’ll use the color for), and I’ll suggest some you might love.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1178, 'prompt_tokens': 12, 'total_tokens': 1190, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1088, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZog978z3x9aUvm

## Accessing Context

In [6]:
from langchain.tools import tool, ToolRuntime


@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour


@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [7]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext,
)

In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

pprint(response)

/Users/bohaoli/Desktop/tuto/tuto_langchain/official/001-foundation-langraph/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='2a16b64a-8b8b-4854-b0ea-c5e08c7684fc'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 149, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZosmQ2f1IgEymjiD7CbvoFjWNcPi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd70c-f023-7f22-af88-8c7ed8aea275-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_OW7ttOVwGC8jRte3EZFe7H5h', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green"),
)

pprint(response)

/Users/bohaoli/Desktop/tuto/tuto_langchain/official/001-foundation-langraph/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='5220e357-64ee-413c-9150-1d36bf7f2883'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 149, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZp0FSyE96NhiCVPRkvm4w6AVnvpj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd714-0690-7e22-b5c7-f6192acbea9e-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_OIuGVlf2vbGflkiNiezxD3Sr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_